# A/B Test: E-Commerce Landing Page Conversion Experiment

**Goal:** Determine whether the new landing page (`new_page`) yields a higher conversion rate than the old landing page (`old_page`).

**Dataset:** `ab_test_raw.csv` — ~294,478 users randomly assigned to control or treatment.

**Column mapping:**
| Column | Meaning |
|---|---|
| `id` | User identifier |
| `time` | Time of visit |
| `con_treat` | Group assignment (`control` / `treatment`) |
| `page` | Landing page shown (`old_page` / `new_page`) |
| `converted` | Whether the user converted (1 = yes, 0 = no) |

---
**Hypotheses:**

- **H₀:** Conversion rate (treatment) = Conversion rate (control)
- **H₁:** Conversion rate (treatment) > Conversion rate (control)  *(one-tailed, right-sided)*

**Significance level:** α = 0.05

## 1. Imports and Data Load

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")

# Load data (path relative to notebook location)
DATA_PATH = Path("../data/ab_test_raw.csv")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

In [ ]:
print("Columns:", df.columns.tolist())
print("\nDtype summary:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

## 2. Data Cleaning

Remove any mismatched rows where `con_treat` and `page` do not align as expected
(control should see old_page; treatment should see new_page).

In [ ]:
print("Group × Page cross-tab (before cleaning):")
print(pd.crosstab(df["con_treat"], df["page"]))

# Keep only correctly assigned rows
df_clean = df[
    ((df["con_treat"] == "control")   & (df["page"] == "old_page")) |
    ((df["con_treat"] == "treatment") & (df["page"] == "new_page"))
].copy()

print(f"\nRows removed (mismatch): {len(df) - len(df_clean)}")
print(f"Clean dataset shape: {df_clean.shape}")

print("\nGroup × Page cross-tab (after cleaning):")
print(pd.crosstab(df_clean["con_treat"], df_clean["page"]))

## 3. Exploratory Data Analysis

In [ ]:
# Conversion rate by group
conv_rates = df_clean.groupby("con_treat")["converted"].agg(
    conversions="sum",
    total="count",
    conversion_rate="mean"
)
conv_rates["conversion_rate_pct"] = conv_rates["conversion_rate"] * 100
print(conv_rates)
conv_rates

In [ ]:
# Bar chart of conversion rates
fig, ax = plt.subplots(figsize=(7, 4))
groups = conv_rates.index.tolist()
rates = conv_rates["conversion_rate_pct"].values

bars = ax.bar(groups, rates, color=["#4C72B0", "#DD8452"], width=0.5, edgecolor="black")
ax.bar_label(bars, fmt="%.2f%%", padding=3, fontsize=11)
ax.set_ylim(0, max(rates) * 1.3)
ax.set_xlabel("Group", fontsize=12)
ax.set_ylabel("Conversion Rate (%)", fontsize=12)
ax.set_title("Conversion Rate: Control vs Treatment", fontsize=14)
plt.tight_layout()
plt.savefig("../reports/conversion_rates.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Sample balance check
fig, ax = plt.subplots(figsize=(6, 4))
conv_rates["total"].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"], edgecolor="black", rot=0)
ax.set_title("Sample Sizes per Group", fontsize=13)
ax.set_xlabel("Group")
ax.set_ylabel("Number of Users")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Two-Proportion Z-Test

In [ ]:
# Extract values
converted_control = int(df_clean.loc[df_clean["con_treat"] == "control",  "converted"].sum())
converted_treat   = int(df_clean.loc[df_clean["con_treat"] == "treatment", "converted"].sum())

n_control = int((df_clean["con_treat"] == "control").sum())
n_treat   = int((df_clean["con_treat"] == "treatment").sum())

print(f"Control   : n={n_control:,}, conversions={converted_control:,}")
print(f"Treatment : n={n_treat:,}, conversions={converted_treat:,}")

In [ ]:
# Two-proportion z-test (H1: treatment > control, one-tailed)
count = np.array([converted_treat, converted_control])
nobs  = np.array([n_treat, n_control])

stat, pval = proportions_ztest(count, nobs, alternative="larger")
print(f"Z-statistic : {stat:.4f}")
print(f"P-value     : {pval:.4f}")
print()
if pval < 0.05:
    print("✅ Reject H₀ at α=0.05: the new page converts significantly better.")
else:
    print("❌ Fail to reject H₀ at α=0.05: no significant lift from the new page.")

## 5. Confidence Intervals

In [ ]:
p_control = converted_control / n_control
p_treat   = converted_treat   / n_treat
diff      = p_treat - p_control

# 95% CI for each proportion
ci_control = proportion_confint(converted_control, n_control, alpha=0.05, method="normal")
ci_treat   = proportion_confint(converted_treat,   n_treat,   alpha=0.05, method="normal")

# 95% CI for the difference (normal approximation)
se_diff = np.sqrt(p_treat * (1 - p_treat) / n_treat +
                  p_control * (1 - p_control) / n_control)
diff_ci = (diff - 1.96 * se_diff, diff + 1.96 * se_diff)

print(f"Control   conversion: {p_control:.4f}  95% CI: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Treatment conversion: {p_treat:.4f}  95% CI: [{ci_treat[0]:.4f}, {ci_treat[1]:.4f}]")
print()
print(f"Difference (treat − control): {diff:+.6f} ({diff*100:+.4f} pp)")
print(f"95% CI for difference:        [{diff_ci[0]*100:.3f} pp, {diff_ci[1]*100:.3f} pp]")

In [ ]:
# Visualise the CI for the difference
fig, ax = plt.subplots(figsize=(8, 3))
ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="No difference (H₀)")
ax.errorbar(
    x=[diff * 100],
    y=[0],
    xerr=[[abs(diff_ci[0] * 100 - diff * 100)],
          [abs(diff_ci[1] * 100 - diff * 100)]],
    fmt="o",
    color="steelblue",
    capsize=8,
    markersize=8,
    label="Observed difference ± 95% CI"
)
ax.set_xlabel("Difference in Conversion Rate (percentage points)", fontsize=11)
ax.set_yticks([])
ax.set_title("95% CI for Difference in Conversion Rate (Treatment − Control)", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("../reports/ci_difference.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Business Interpretation

| Metric | Control | Treatment |
|---|---|---|
| Sample size | 145,274 | 145,311 |
| Conversions | 17,489 | 17,264 |
| Conversion rate | 12.04% | 11.89% |

**Statistical result:**  
Z = −1.31, p-value = 0.905 (one-tailed).  
We **fail to reject H₀** at α = 0.05.

**95% CI for the difference** (treatment − control): [−0.394 pp, +0.078 pp]  
The interval contains zero, confirming the absence of a statistically significant lift.

**Recommendation:**  
The new landing page does **not** improve conversion rates. With ~147K users per arm, the test is well-powered; the true effect is negligible. **Do not ship the new page.** Consider deeper UX research or testing a more differentiated design.